# Notebook 51 — Chronos Embedding Composition Test

**Does the Chronos-T5-Small foundation model's embedding space encode composition structure geometrically?**

nb50 showed that the nb46 composition-trained transformer's embedding midpoints score 32.8% — *worse* than the 6f centroid midpoint (45.3%). Its attention mechanism reaches 93.8%, but the geometry is actively anti-compositional (midpoints collapse to burst, a geometric hub).

nb51 asks the same question of **Chronos-T5-Small** (46M params, 512-d encoder, pretrained on 27B time-series points with no composition task). Two angles:

1. **Embedding midpoint:** Average the two class centroid embeddings, classify to nearest centroid. Does massive pretraining give Chronos richer shape geometry that is more composable than the task-trained transformer?
2. **Actual-mean embedding:** For N mixed signals per pair, extract Chronos embedding for each, average → nearest centroid. Does Chronos "see" the same composition outcomes as the 6f classifier, or does its richer representation give different composition predictions?

---

## Pre-run predictions

**F173:** ρ(Chronos emb distance, fingerprint distance) > 0.399 (task-trained transformer from nb46/nb50). Chronos was pretrained on vastly more diverse signals and should have richer shape-similarity geometry.

**F174:** Chronos embedding midpoint accuracy ≈ 40–55% — similar to or slightly above the 6f midpoint (45.3%). No composition training → no composition geometry in midpoints. But Chronos representations are richer, so midpoints may be more "signal-like" than the task-trained transformer's collapsed geometry.

**F175:** Chronos actual-mean accuracy < 96.9% (simulation oracle). Chronos and the 6f classifier may disagree on how to classify some mixed signals — particularly the marginal/ambiguous pairs near class boundaries.

**F176:** Chronos actual-mean accuracy > Chronos embedding midpoint. The mean over actual mixed-signal embeddings should outperform the centroid midpoint, mirroring the nb48 simulation result in the 6f space.

In [1]:
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import torch
from chronos import ChronosPipeline
import time, sys
sys.path.insert(0, '..')

SIGNED_COLS = ['skewness', 'kurtosis', 'lag1_autocorr', 'zero_crossings', 'slope', 'baseline_delta']
SEQ_LEN = 64; SEED = 42; t64 = np.linspace(0, 1, SEQ_LEN)

def zscore(s):
    s = np.asarray(s, dtype=float); std = s.std()
    return (s - s.mean()) / std if std > 1e-8 else s * 0.0

def baseline_delta_fn(s, frac=0.10):
    k = max(1, int(len(s) * frac))
    return float(np.mean(s[-k:]) - np.mean(s[:k]))

def extract_6f(s):
    arr = np.asarray(s, dtype=float); t = np.arange(len(arr))
    lag1 = float(np.corrcoef(arr[:-1], arr[1:])[0, 1]) if len(arr) > 2 else 0.0
    return {
        'skewness':       float(stats.skew(arr)),
        'kurtosis':       float(stats.kurtosis(arr)),
        'lag1_autocorr':  lag1,
        'zero_crossings': float(np.sum(np.diff(np.sign(arr)) != 0) / len(arr)),
        'slope':          float(stats.linregress(t, arr).slope),
        'baseline_delta': baseline_delta_fn(arr),
    }

GENERATORS = {
    'burst':              lambda r: zscore(np.exp(-(t64-r.uniform(.15,.50))**2/(2*r.uniform(.05,.15)**2))+r.normal(0,.05,SEQ_LEN)),
    'oscillator':         lambda r: zscore(np.sin(2*np.pi*r.uniform(1.5,4.5)*t64+r.uniform(0,np.pi))+r.normal(0,.05,SEQ_LEN)),
    'seasonal':           lambda r: zscore(np.sin(2*np.pi*r.uniform(3,6)*t64)+.25*np.sin(4*np.pi*r.uniform(3,6)*t64)+r.normal(0,.04,SEQ_LEN)),
    'trend':              lambda r: zscore(t64+r.uniform(.05,.30)*t64**2+r.normal(0,.02,SEQ_LEN)),
    'integrated_trend':   lambda r: zscore(np.cumsum(np.ones(SEQ_LEN)*r.uniform(.015,.035)+r.normal(0,.003,SEQ_LEN))),
    'irregular_osc':      lambda r: zscore((np.sin(2*np.pi*r.uniform(2,5)*t64)*(1+r.uniform(.3,.8,SEQ_LEN))+r.normal(0,.3,SEQ_LEN))*1.4),
    'declining_osc':      lambda r: zscore(np.linspace(r.uniform(.9,1.2),r.uniform(.35,.65),SEQ_LEN)*np.sin(2*np.pi*r.uniform(2.5,5.5)*t64)+np.linspace(0,r.uniform(-.8,-.4),SEQ_LEN)+r.normal(0,.05,SEQ_LEN)),
    'declining_monotonic':lambda r: zscore(np.cumsum(-np.ones(SEQ_LEN)*r.uniform(.015,.035)+r.normal(0,.003,SEQ_LEN))),
}

CLASSES = list(GENERATORS.keys())
N_CLASSES = len(CLASSES)
CLS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
ABBREV = {
    'burst': 'BUR', 'oscillator': 'OSC', 'seasonal': 'SEA',
    'trend': 'TRE', 'integrated_trend': 'INT', 'irregular_osc': 'IRR',
    'declining_osc': 'DCO', 'declining_monotonic': 'DCM',
}

# 6-feature centroid classifier
recs = []
for cls, gen in GENERATORS.items():
    for i in range(200):
        r = np.random.default_rng(SEED + CLASSES.index(cls)*1000 + i)
        f = extract_6f(gen(r)); f['class'] = cls; recs.append(f)
df_fp = pd.DataFrame(recs)
sc = StandardScaler()
X_fp = sc.fit_transform(df_fp[SIGNED_COLS].values)
ctrds_6f = {c: X_fp[df_fp['class']==c].mean(axis=0) for c in GENERATORS}

def classify_6f(feat_dict):
    x = sc.transform([[feat_dict[c] for c in SIGNED_COLS]])[0]
    dists = {c: float(np.linalg.norm(x - v)) for c, v in ctrds_6f.items()}
    return min(dists, key=dists.get), dists

print('6-feature classifier ready.')

# Load Chronos
print('Loading Chronos-T5-Small ...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
pipeline = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map=device,
    dtype=torch.float32,
)
encoder = pipeline.model.model.encoder
encoder.eval()
print(f'Chronos loaded. Hidden dim: {encoder.config.d_model}  device: {device}')

I0000 00:00:1777806319.883160   64099 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777806320.082217   64099 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1777806320.835719   64099 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777806320.837004   64099 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


6-feature classifier ready.
Loading Chronos-T5-Small ...


Chronos loaded. Hidden dim: 512  device: cuda


In [2]:
# ---- Chronos embedding function (batched) ----

def get_chronos_embeddings_batch(signals, batch_size=32):
    """
    Extract mean-pooled T5 encoder embeddings for a list of 1-D signals.
    signals: list of numpy arrays (each length SEQ_LEN, z-scored)
    Returns: (N, D) numpy array
    """
    all_embs = []
    for start in range(0, len(signals), batch_size):
        batch = signals[start:start+batch_size]
        # Stack into (B, T) tensor
        ts = torch.tensor(np.stack(batch), dtype=torch.float32)  # (B, T)
        token_ids, attention_mask, _ = pipeline.tokenizer.context_input_transform(ts)
        token_ids     = token_ids.to(device)
        attention_mask = attention_mask.to(device)
        with torch.no_grad():
            enc_out = encoder(input_ids=token_ids, attention_mask=attention_mask)
        h    = enc_out.last_hidden_state                           # (B, seq_len, D)
        mask = attention_mask.unsqueeze(-1).float()
        emb  = (h * mask).sum(dim=1) / mask.sum(dim=1)           # (B, D)
        all_embs.append(emb.cpu().numpy())
    return np.vstack(all_embs)

# ---- Build composition table (same seeds as nb46/nb50) ----
N_SAMPLES_PER_PAIR = 500

table = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
table_purity = np.zeros((N_CLASSES, N_CLASSES))
table_name = [['' for _ in range(N_CLASSES)] for _ in range(N_CLASSES)]

print('Deriving composition table (500 samples/pair) ...')
t0 = time.time()
for i, cls_a in enumerate(CLASSES):
    for j, cls_b in enumerate(CLASSES):
        results = []
        for k in range(N_SAMPLES_PER_PAIR):
            r_a = np.random.default_rng(1000 + i*5000 + k)
            r_b = np.random.default_rng(2000 + j*5000 + k)
            mixed = zscore(0.5 * GENERATORS[cls_a](r_a) + 0.5 * GENERATORS[cls_b](r_b))
            cls_out, _ = classify_6f(extract_6f(mixed))
            results.append(cls_out)
        cnt = Counter(results)
        top_cls, top_n = cnt.most_common(1)[0]
        table[i, j]       = CLS_TO_IDX[top_cls]
        table_purity[i,j] = top_n / N_SAMPLES_PER_PAIR
        table_name[i][j]  = top_cls
print(f'Done in {time.time()-t0:.1f}s')

# Baseline accuracies
correct_midpt_6f = 0
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        mid = (ctrds_6f[CLASSES[i]] + ctrds_6f[CLASSES[j]]) / 2
        pred = min(ctrds_6f, key=lambda c: np.linalg.norm(mid - ctrds_6f[c]))
        if pred == table_name[i][j]: correct_midpt_6f += 1
acc_6f_midpt = correct_midpt_6f / 64
print(f'6f centroid midpoint baseline: {acc_6f_midpt:.3f} ({correct_midpt_6f}/64)')

Deriving composition table (500 samples/pair) ...


Done in 15.6s
6f centroid midpoint baseline: 0.453 (29/64)


In [3]:
# ---- Build Chronos class centroid embeddings ----
# N_CENTROID signals per class → mean-pool Chronos embeddings → class centroid

N_CENTROID = 100

print(f'Extracting Chronos embeddings for {N_CENTROID} signals × {N_CLASSES} classes ...')
t0 = time.time()

chronos_centroids = {}   # class → (D,) mean embedding
chronos_centroid_std = {}  # class → (D,) std (spread within class)

for cls_idx, cls in enumerate(CLASSES):
    gen = GENERATORS[cls]
    signals = [gen(np.random.default_rng(SEED + cls_idx*1000 + k)) for k in range(N_CENTROID)]
    embs = get_chronos_embeddings_batch(signals, batch_size=32)  # (N_CENTROID, D)
    chronos_centroids[cls]    = embs.mean(axis=0)
    chronos_centroid_std[cls] = embs.std(axis=0)
    print(f'  {cls:20s}: emb_norm={np.linalg.norm(chronos_centroids[cls]):.3f}  '
          f'intra_std={chronos_centroid_std[cls].mean():.4f}')

D = len(chronos_centroids[CLASSES[0]])
print(f'\nChronos embedding dim: {D}')
print(f'Total time: {time.time()-t0:.1f}s')

# Pairwise centroid distances
ctrd_emb_arr = np.array([chronos_centroids[c] for c in CLASSES])  # (8, D)
chronos_ctrd_dists = squareform(pdist(ctrd_emb_arr, metric='euclidean'))
print('\nChronos pairwise centroid distances (top 5 closest pairs):')
pairs_sorted = sorted(
    [(chronos_ctrd_dists[i,j], CLASSES[i], CLASSES[j])
     for i in range(N_CLASSES) for j in range(i+1, N_CLASSES)]
)
for d, a, b in pairs_sorted[:5]:
    print(f'  {a:22s} ↔ {b:22s}: {d:.4f}')
print('Top 5 furthest:')
for d, a, b in pairs_sorted[-5:][::-1]:
    print(f'  {a:22s} ↔ {b:22s}: {d:.4f}')

Extracting Chronos embeddings for 100 signals × 8 classes ...
  burst               : emb_norm=0.246  intra_std=0.0046


  oscillator          : emb_norm=0.228  intra_std=0.0045


  seasonal            : emb_norm=0.219  intra_std=0.0046
  trend               : emb_norm=0.260  intra_std=0.0028
  integrated_trend    : emb_norm=0.263  intra_std=0.0008
  irregular_osc       : emb_norm=0.224  intra_std=0.0041
  declining_osc       : emb_norm=0.213  intra_std=0.0034
  declining_monotonic : emb_norm=0.258  intra_std=0.0007

Chronos embedding dim: 512
Total time: 0.4s

Chronos pairwise centroid distances (top 5 closest pairs):
  oscillator             ↔ seasonal              : 0.0909
  integrated_trend       ↔ declining_monotonic   : 0.0957
  seasonal               ↔ declining_osc         : 0.1094
  seasonal               ↔ irregular_osc         : 0.1112
  oscillator             ↔ irregular_osc         : 0.1269
Top 5 furthest:
  burst                  ↔ integrated_trend      : 0.3092
  burst                  ↔ declining_monotonic   : 0.3027
  integrated_trend       ↔ irregular_osc         : 0.3000
  oscillator             ↔ integrated_trend      : 0.2949
  irregular_osc

In [4]:
# ---- Part A: Embedding midpoint test ----

def classify_by_chronos_centroid(vec):
    dists = {c: float(np.linalg.norm(vec - chronos_centroids[c])) for c in CLASSES}
    return min(dists, key=dists.get), dists

correct_ch_midpt = 0
ch_midpt_preds = np.zeros((N_CLASSES, N_CLASSES), dtype=int)

print('=== Part A: Chronos embedding midpoint ===\n')
print(f'{"Pair":12s}  {"Empirical":12s}  {"Ch-midpt":12s}  {"✓?":4s}')
print('-' * 48)

for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        midpt_vec = (chronos_centroids[CLASSES[i]] + chronos_centroids[CLASSES[j]]) / 2
        pred_cls, _ = classify_by_chronos_centroid(midpt_vec)
        pred_idx = CLS_TO_IDX[pred_cls]
        ch_midpt_preds[i, j] = pred_idx
        ok = pred_cls == table_name[i][j]
        if ok: correct_ch_midpt += 1
        print(f'{ABBREV[CLASSES[i]]},{ABBREV[CLASSES[j]]:12s}  '
              f'{ABBREV[table_name[i][j]]:12s}  {ABBREV[pred_cls]:12s}  {"✓" if ok else "✗"}')

acc_ch_midpt = correct_ch_midpt / 64
print(f'\n{"Method":45s}  {"Accuracy":>10s}  {"Correct":>8s}')
print('-' * 70)
print(f'{"6f centroid midpoint (nb47)":45s}  {acc_6f_midpt:10.1%}  {int(acc_6f_midpt*64):>8d}/64')
print(f'{"Transformer emb midpoint (nb50)":45s}  {"32.8%":>10s}  {"21":>8s}/64')
print(f'{"Chronos emb midpoint (nb51 THIS RESULT)":45s}  {acc_ch_midpt:10.1%}  {correct_ch_midpt:>8d}/64')
print(f'{"Closed-form linear ceiling (nb49)":45s}  {"70.3%":>10s}  {"45":>8s}/64')
print(f'{"Simulation oracle (nb48)":45s}  {"96.9%":>10s}  {"62":>8s}/64')

# Which class dominates Chronos midpoint predictions?
print('\nChronos midpoint prediction distribution:')
flat_midpt = ch_midpt_preds.flatten()
flat_emp   = table.flatten()
for cls, cnt in sorted(Counter(CLASSES[k] for k in flat_midpt).items(), key=lambda x: -x[1]):
    emp_cnt = sum(1 for k in flat_emp if CLASSES[k] == cls)
    print(f'  {cls:20s}: predicted {cnt:3d}x  (empirical: {emp_cnt:3d}x)')

=== Part A: Chronos embedding midpoint ===

Pair          Empirical     Ch-midpt      ✓?  
------------------------------------------------
BUR,BUR           BUR           BUR           ✓
BUR,OSC           DCO           BUR           ✗
BUR,SEA           DCO           IRR           ✗
BUR,TRE           INT           BUR           ✗
BUR,INT           INT           BUR           ✗
BUR,IRR           DCO           BUR           ✗
BUR,DCO           DCO           BUR           ✗
BUR,DCM           DCM           BUR           ✗
OSC,BUR           DCO           BUR           ✗
OSC,OSC           OSC           OSC           ✓
OSC,SEA           DCO           OSC           ✗
OSC,TRE           TRE           OSC           ✗
OSC,INT           TRE           DCO           ✗
OSC,IRR           SEA           OSC           ✗
OSC,DCO           DCO           OSC           ✗
OSC,DCM           DCO           DCO           ✓
SEA,BUR           DCO           IRR           ✗
SEA,OSC           DCO           OSC         

In [5]:
# ---- Part B: Actual-mean Chronos embedding test ----
# For each (i,j) pair: generate N_MIX mixed signals, extract Chronos embedding,
# average → nearest class centroid → composition prediction

N_MIX = 100

print(f'=== Part B: Chronos actual-mean embedding ({N_MIX} mixed signals/pair) ===')
print(f'Total Chronos passes: {N_MIX * 64}  (estimated time: depends on GPU speed)')
t0 = time.time()

correct_ch_actual = 0
ch_actual_preds = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
ch_actual_mean_embs = {}  # (i,j) → (D,) mean embedding

for i, cls_a in enumerate(CLASSES):
    for j, cls_b in enumerate(CLASSES):
        mixed_signals = []
        for k in range(N_MIX):
            r_a = np.random.default_rng(1000 + i*5000 + k)
            r_b = np.random.default_rng(2000 + j*5000 + k)
            mixed = zscore(0.5 * GENERATORS[cls_a](r_a) + 0.5 * GENERATORS[cls_b](r_b))
            mixed_signals.append(mixed)
        embs = get_chronos_embeddings_batch(mixed_signals, batch_size=32)  # (N_MIX, D)
        mean_emb = embs.mean(axis=0)
        ch_actual_mean_embs[(i, j)] = mean_emb
        pred_cls, _ = classify_by_chronos_centroid(mean_emb)
        ch_actual_preds[i, j] = CLS_TO_IDX[pred_cls]
        if pred_cls == table_name[i][j]: correct_ch_actual += 1

acc_ch_actual = correct_ch_actual / 64
elapsed = time.time() - t0
print(f'\nDone in {elapsed:.1f}s  ({elapsed/64:.1f}s per pair)')

print(f'\n{"Method":45s}  {"Accuracy":>10s}  {"Correct":>8s}')
print('-' * 70)
print(f'{"6f centroid midpoint (nb47)":45s}  {acc_6f_midpt:10.1%}  {int(acc_6f_midpt*64):>8d}/64')
print(f'{"Transformer emb midpoint (nb50)":45s}  {"32.8%":>10s}  {"21":>8s}/64')
print(f'{"Chronos emb midpoint (nb51)":45s}  {acc_ch_midpt:10.1%}  {correct_ch_midpt:>8d}/64')
print(f'{"Chronos actual-mean emb (nb51)":45s}  {acc_ch_actual:10.1%}  {correct_ch_actual:>8d}/64')
print(f'{"Closed-form linear ceiling (nb49)":45s}  {"70.3%":>10s}  {"45":>8s}/64')
print(f'{"Transformer forward pass (nb50)":45s}  {"93.8%":>10s}  {"60":>8s}/64')
print(f'{"Simulation oracle (nb48)":45s}  {"96.9%":>10s}  {"62":>8s}/64')

print(f'\nGap (actual-mean vs midpoint): {acc_ch_actual - acc_ch_midpt:+.1%}')

# Actual-mean prediction distribution
print('\nChronos actual-mean prediction distribution:')
flat_actual = ch_actual_preds.flatten()
flat_emp    = table.flatten()
for cls, cnt in sorted(Counter(CLASSES[k] for k in flat_actual).items(), key=lambda x: -x[1]):
    emp_cnt = sum(1 for k in flat_emp if CLASSES[k] == cls)
    print(f'  {cls:20s}: predicted {cnt:3d}x  (empirical: {emp_cnt:3d}x)')

=== Part B: Chronos actual-mean embedding (100 mixed signals/pair) ===
Total Chronos passes: 6400  (estimated time: depends on GPU speed)



Done in 1.8s  (0.0s per pair)

Method                                           Accuracy   Correct
----------------------------------------------------------------------
6f centroid midpoint (nb47)                         45.3%        29/64
Transformer emb midpoint (nb50)                     32.8%        21/64
Chronos emb midpoint (nb51)                         26.6%        17/64
Chronos actual-mean emb (nb51)                      37.5%        24/64
Closed-form linear ceiling (nb49)                   70.3%        45/64
Transformer forward pass (nb50)                     93.8%        60/64
Simulation oracle (nb48)                            96.9%        62/64

Gap (actual-mean vs midpoint): +10.9%

Chronos actual-mean prediction distribution:
  declining_osc       : predicted  30x  (empirical:  25x)
  irregular_osc       : predicted  23x  (empirical:   4x)
  trend               : predicted   7x  (empirical:   5x)
  burst               : predicted   1x  (empirical:   1x)
  seasonal     

In [6]:
# ---- Part C: Embedding geometry analysis ----
# ρ(Chronos centroid dist, fingerprint dist) vs ρ(Chronos centroid dist, comp impurity)

ctrd_6f_arr = np.array([ctrds_6f[c] for c in CLASSES])
fp_dists    = squareform(pdist(ctrd_6f_arr, metric='euclidean'))
comp_impurity = 1.0 - table_purity

mask = np.triu(np.ones((N_CLASSES, N_CLASSES), dtype=bool), k=1)
ch_dist_flat    = chronos_ctrd_dists[mask]
fp_flat         = fp_dists[mask]
comp_impure_flat = comp_impurity[mask]

rho_fp,   pval_fp   = spearmanr(ch_dist_flat, fp_flat)
rho_comp, pval_comp = spearmanr(ch_dist_flat, comp_impure_flat)

print('=== Part C: Chronos embedding geometry ===')
print(f'\nSpearman ρ(Chronos ↔ fingerprint dist):   {rho_fp:+.3f}  (p={pval_fp:.4f})')
print(f'Task-trained transformer (nb46/nb50):      +0.399')
print(f'\nSpearman ρ(Chronos ↔ comp impurity):      {rho_comp:+.3f}  (p={pval_comp:.4f})')
print(f'Task-trained transformer (nb50):           +0.175 (p=0.37)')

# Intra-class vs inter-class spread
print('\nIntra-class embedding spread (mean std per class):')
for cls in CLASSES:
    print(f'  {cls:20s}: mean_std = {chronos_centroid_std[cls].mean():.4f}')

# Compare Chronos vs 6f centroid distances: do the same class pairs rank similarly?
print('\nClass pair distance comparison (Chronos vs 6f fingerprint):')
pairs = [(i,j) for i in range(N_CLASSES) for j in range(i+1,N_CLASSES)]
pair_data = [(chronos_ctrd_dists[i,j], fp_dists[i,j], CLASSES[i], CLASSES[j]) for i,j in pairs]
pair_data.sort(key=lambda x: x[0])
print(f'{"Pair":40s}  {"Chronos":>10s}  {"6f-fp":>8s}')
for cd, fd, a, b in pair_data:
    print(f'  {a[:10]:10s} ↔ {b[:10]:10s}{"":15s}  {cd:10.4f}  {fd:8.4f}')

=== Part C: Chronos embedding geometry ===

Spearman ρ(Chronos ↔ fingerprint dist):   +0.304  (p=0.1154)
Task-trained transformer (nb46/nb50):      +0.399

Spearman ρ(Chronos ↔ comp impurity):      -0.233  (p=0.2324)
Task-trained transformer (nb50):           +0.175 (p=0.37)

Intra-class embedding spread (mean std per class):
  burst               : mean_std = 0.0046
  oscillator          : mean_std = 0.0045
  seasonal            : mean_std = 0.0046
  trend               : mean_std = 0.0028
  integrated_trend    : mean_std = 0.0008
  irregular_osc       : mean_std = 0.0041
  declining_osc       : mean_std = 0.0034
  declining_monotonic : mean_std = 0.0007

Class pair distance comparison (Chronos vs 6f fingerprint):
Pair                                         Chronos     6f-fp
  oscillator ↔ seasonal                       0.0909    1.5796
  integrated ↔ declining_                     0.0957    4.2158
  seasonal   ↔ declining_                     0.1094    1.4976
  seasonal   ↔ irregula

In [7]:
# ---- Part D: Visualization ----

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Full accuracy ladder
methods = ['Transformer\nemb midpt\n(nb50)', '6f midpt\n(nb47)',
           'Chronos\nemb midpt\n(nb51)', 'Chronos\nactual-mean\n(nb51)',
           'Closed-form\n(nb49)', 'Transformer\nfwd pass\n(nb50)', 'Simulation\n(nb48)']
accs    = [0.328, acc_6f_midpt, acc_ch_midpt, acc_ch_actual, 0.703, 0.938, 0.969]
colors  = ['#FF7043', '#aaaaaa', '#AB47BC', '#7B1FA2', '#888888', '#2196F3', '#4CAF50']
bars = axes[0].bar(methods, [a*100 for a in accs], color=colors, alpha=0.85, edgecolor='white')
axes[0].axhline(70.3, color='gray', lw=1.2, ls='--', alpha=0.7)
axes[0].set_ylabel('Composition-table accuracy (%)')
axes[0].set_ylim(0, 107)
axes[0].set_title('Complete accuracy ladder\n(nb47–nb51)', fontsize=10)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{acc:.1%}', ha='center', va='bottom', fontsize=7)
axes[0].tick_params(axis='x', labelsize=7)

# Panel 2: Chronos centroid distance heatmap
im = axes[1].imshow(chronos_ctrd_dists, cmap='Blues')
axes[1].set_xticks(range(N_CLASSES)); axes[1].set_xticklabels([ABBREV[c] for c in CLASSES], rotation=45, fontsize=8)
axes[1].set_yticks(range(N_CLASSES)); axes[1].set_yticklabels([ABBREV[c] for c in CLASSES], fontsize=8)
axes[1].set_title('Chronos centroid distances\n(512-d T5 encoder)', fontsize=10)
plt.colorbar(im, ax=axes[1])

# Panel 3: Chronos dist vs 6f fingerprint dist scatter
axes[2].scatter(fp_flat, ch_dist_flat, alpha=0.65, color='#AB47BC', s=45)
axes[2].set_xlabel('6f fingerprint distance')
axes[2].set_ylabel('Chronos centroid distance (512-d)')
axes[2].set_title(f'Chronos vs fingerprint similarity\nSpearman ρ = {rho_fp:+.3f}', fontsize=10)
m, b_ = np.polyfit(fp_flat, ch_dist_flat, 1)
xs = np.linspace(fp_flat.min(), fp_flat.max(), 50)
axes[2].plot(xs, m*xs + b_, 'k--', lw=1.5, alpha=0.7)

fig.suptitle('Notebook 51 — Chronos Embedding Composition Test', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../artifacts/nb51_chronos_embedding.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


---
## Findings — Notebook 51

### F173 — ρ(Chronos, fingerprint): +0.304 (p=0.115) — refuted; lower than task-trained transformer and not significant

**Prediction:** > 0.399. **Refuted — lower, not higher.**

The Chronos 512-d embedding space correlates *less* with the 6f fingerprint geometry than the task-trained 128-d transformer. ρ=+0.304 is not statistically significant (p=0.115). The closest Chronos class pairs are oscillator↔seasonal (0.091) and integrated_trend↔declining_monotonic (0.096) — both far apart in 6f space (1.58 and 4.22 respectively). Chronos has learned a different notion of shape similarity, likely weighting trend/level features over oscillatory structure.

---

### F174 — Chronos embedding midpoint: 26.6% (17/64) — refuted; worse than both baselines

**Prediction:** 40–55%. **Refuted — 26.6%, the worst result of all methods tested.**

Chronos midpoints perform worse than the task-trained transformer midpoints (32.8%) and the 6f midpoint (45.3%). Both tasks of embedding midpoint prediction have now been conclusively refuted across all three representation spaces (6-d, 128-d, 512-d). The Chronos midpoints collapse with a different attractor: oscillator and declining_osc are over-predicted; seasonal and integrated_trend are severely under-predicted (1x vs 8x and 7x empirically).

---

### F175 — Chronos actual-mean: 37.5% (24/64) — confirmed < 96.9%, but surprisingly low

**Prediction:** < 96.9% (simulation oracle). **Trivially confirmed, but the actual number is striking.**

Even averaging 100 Chronos embeddings of actual mixed signals per pair yields only 37.5% — *worse* than the naive 6f centroid midpoint (45.3%). Chronos fundamentally disagrees with the 6f classifier on how to categorize mixed signals: it massively over-predicts irregular_osc (23x predicted vs 4x empirical) and under-predicts seasonal (1x vs 8x) and integrated_trend (1x vs 7x). Chronos perceives mixed oscillatory signals as irregular where the 6f classifier sees them as seasonal or integrated.

---

### F176 — actual-mean > embedding midpoint: +10.9pp — confirmed

**Prediction:** Confirmed. **37.5% vs 26.6%, gap = +10.9pp.**

The gap is much smaller than in the 6f space (where actual-mean = 96.9% vs midpoint = 45.3%, gap = 51.6pp). The narrower gap in Chronos space reflects the same principle: averaging over actual mixed signals is better than interpolating centroids, but the Chronos representation space does not amplify the benefit the way the 6f classifier does.

---

### F177 — Emergent: Chronos actual-mean (37.5%) < 6f midpoint (45.3%) — the classifier gap

Averaging 100 Chronos embeddings of actual mixed signals per pair still underperforms the simplest geometric method in 6f space. This is the cross-receiver comparison: Chronos and the 6f classifier do not agree on what mixed signals are. The 6f classifier is calibrated to the empirical composition table (which was derived using the 6f classifier). Chronos is an independent receiver with a different inductive bias — it sees shape features that the 6f statistics miss, and its classifications reflect those different features.

**Implication:** The empirical composition table is 6f-classifier-relative. It measures what a specific feature extractor produces on mixed signals. A different receiver (Chronos) would produce a *different* composition table. This is Phase 3's observer-relativity in action at the composition level.

---

### F178 — Emergent: Chronos embedding geometry is orthogonal to composition structure (ρ = -0.233, p=0.23)

ρ(Chronos, composition impurity) = -0.233 — slightly *negative* and not significant. Closer Chronos embeddings tend to yield *more* impure compositions — the opposite of what composition-predictive geometry would look like. Chronos's shape similarity structure (oscillator≈seasonal, integrated_trend≈declining_monotonic) is orthogonal to composition structure (mixing oscillator+seasonal → declining_osc, not their average).

---

### F179 — Emergent: Complete accuracy ladder for Thread 2

| Method | Accuracy | Receiver |
|---|---|---|
| Chronos emb midpoint (nb51) | **26.6%** | Chronos 512-d |
| Transformer emb midpoint (nb50) | **32.8%** | Transformer 128-d |
| Chronos actual-mean (nb51) | **37.5%** | Chronos 512-d |
| 6f centroid midpoint (nb47) | 45.3% | 6f fingerprint |
| Closed-form linear (nb49) | 70.3% | 6f fingerprint |
| Transformer forward pass (nb50) | 93.8% | Transformer attention |
| Simulation oracle (nb48) | **96.9%** | 6f fingerprint |

No embedding space tested achieves >50% composition accuracy through geometric midpoints. The 6f actual-mean simulation is the only method that approaches the oracle. Every learned high-d representation performs *worse* than the low-d fingerprint on this geometric interpolation task.

---

**Thread 2 conclusion:** Composition structure is not stored in embedding geometry — not in 6f space, not in task-trained transformer space, not in Chronos's 512-d pretrained space. It is stored in the mixing operator itself (captured by simulation) or in learned attention mechanisms trained explicitly on the composition task. The grokking-transfer hypothesis (which assumed embedding midpoints are meaningful) requires revision: post-grokking representations may not be composable by interpolation even when the model has learned the composition rule.

---

Findings F173–F179 added. Total findings: **179**.